# MeetBuddy AI - Permanent GPU Transcriber

This version is pre-configured with your **Static Domain**.

### 🚀 Instructions:
1. Go to **Runtime -> Change runtime type**, select **T4 GPU** and save.
2. Paste your **Ngrok Authtoken** below.
3. Click **Run All** (Ctrl+F9).
4. Your MeetBuddy app will connect automatically to the URL printed at the bottom.

In [ ]:
# @title 🔑 Setup Your Connection
NGROK_TOKEN = "" # @param {type:"string"}
NGROK_DOMAIN = "polysyllogistic-in-unrhapsodically.ngrok-free.dev"

print("Installing dependencies... (approx 1 minute)")
!pip install -q faster-whisper fastapi uvicorn python-multipart pyngrok nest-asyncio

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("❌ ERROR: Paste your Ngrok token first!")

In [ ]:
import nest_asyncio
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import uvicorn
from faster_whisper import WhisperModel
import os
import threading
import time
import urllib.request

app = FastAPI()
print("Loading Whisper Model (Large-V3) on GPU...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Model Loaded!")

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...), language: str = Form("en")):
    try:
        path = f"/tmp/{file.filename}"
        with open(path, "wb") as f: f.write(await file.read())
        segments, _ = model.transcribe(path, language=None if language=="auto" else language)
        text = "".join([s.text for s in segments])
        os.remove(path)
        return {"text": text.strip(), "confidence": 0.9}
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

nest_asyncio.apply()
threading.Thread(target=run_uvicorn, daemon=True).start()

time.sleep(5) # Wait for server

try:
    ngrok.kill()
    public_url = ngrok.connect(8000, domain=NGROK_DOMAIN).public_url
    
    print("\n" + "="*60)
    print(f"🚀 PERMANENT SERVER READY!")
    print(f"🔗 URL: {public_url}")
    print("="*60)
    
    while True: time.sleep(100)
except Exception as e:
    print(f"Tunnel Error: {e}")